# Embeddings, MPNet & FAISS — Semantic Search from Scratch

**A 2-hour beginner teaching notebook**

> **Before you start**, make sure you have completed:
> - Weekend 1 — How AI & LLMs Work  
> - Weekend 2 — Prompt Engineering  
> - Weekend 3 — Tool Use  

In this notebook you will:
1. Understand *why* keyword search fails and semantics win  
2. Learn how **MPNet** converts sentences into 768-number vectors  
3. Measure meaning with **cosine similarity**  
4. Search millions of vectors in milliseconds with **FAISS**  
5. Build a working **semantic search engine** end-to-end

> **Disclaimer:** `all-mpnet-base-v2` was trained on ~1 billion sentence pairs using contrastive learning, making it particularly strong at semantic similarity tasks. All models auto-download from Hugging Face on first run (~420 MB total).


In [ ]:

# Install all required libraries (run this once)
# sentence-transformers: loads MPNet and handles pooling automatically
# faiss-cpu: Facebook AI Similarity Search, CPU edition (GPU upgrade available later)
# scikit-learn: provides cosine_similarity helper
# numpy: numerical arrays for embeddings

import sys          # needed to get the path of the current Python interpreter
import subprocess   # used to run pip from inside the notebook

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "sentence-transformers", "faiss-cpu", "scikit-learn", "numpy"],
    check=True   # raise CalledProcessError if pip exits with a non-zero code
)
print("All libraries installed successfully")


---
## Section 1 — Why Keywords Fail & Semantics Win

### The Keyword Problem

Traditional search engines work by **counting word matches**.  
If your query word does not appear in a document, the document is invisible — even if its meaning is identical.

> **Connection to Weekend 1:** You learned that Claude *understands* context. Regular search engines don't — they just count word matches. MPNet + FAISS fix this by making search understand *meaning*.

---

### Real-world Failures of Keyword Search

| User Search | Keyword Search Finds | What They Actually Need | Why Keyword Fails |
|---|---|---|---|
| "How to care for pets" | Only docs containing "pet", "care", "for" | "dogs", "cats", "feeding", "grooming" articles | Synonyms and related concepts are invisible |
| "cheap flights" | Docs with "cheap" OR "flights" | Budget airlines, discount codes, comparison tools | Different ways to express the same idea |
| "ML model performance" | Docs with "ML", "model", "performance" | "accuracy", "precision", "recall", "F1 score" | Domain terminology not matched |
| "broken laptop" | Docs with "broken" AND "laptop" | Support pages, troubleshooting, warranty info | Intent unclear — broken how? |

---

### The Solution: Turn Meaning into Numbers

Instead of matching words, we:
1. **Encode** every document as a vector of numbers (an *embedding*)  
2. **Store** those vectors in a fast index  
3. **Encode** the user's query the same way  
4. **Find** the documents whose vectors point in the same direction  

Documents about similar things end up with vectors that *point in similar directions* — regardless of which exact words were used.

---

### Quick Check

> If someone searches **"How do I teach my dog tricks?"** will a keyword search find a result titled **"Puppy Training Guide"**? Why or why not?  
> *(Think about which words overlap and which don't.)*


In [ ]:

# Section 1 — Keyword search vs semantic search (conceptual demo)
# Shows the keyword-matching limitation with plain Python — no ML libraries needed yet.

print("--- Keyword Search Simulation ---")
print()

documents = [
    "Puppy training guide for beginners",
    "How to teach tricks to young dogs",
    "Dog nutrition and feeding schedules",
    "Cat grooming tips",
    "The history of space exploration",
]

query = "How do I teach my dog tricks?"

query_words = set(query.lower().split())
print(f"Query words: {query_words}")
print()

print("Keyword match scores:")
for document_index, document_text in enumerate(documents):
    document_words = set(document_text.lower().split())
    overlap_count = len(query_words & document_words)
    print(f"  [{overlap_count} matches]  {document_text}")

print()
print("Notice: 'Puppy training guide' (index 0) scores 0 — zero word overlap!")
print("But it is clearly the MOST relevant document.")
print()
print("Semantic search solves this. Let's learn how.")


---
## Section 2 — Understanding MPNet: From Text to 768 Numbers

### What Is MPNet?

**MPNet** is a Transformer-based model that uses **two-stream self-attention**, allowing it to process text using both positional information and masked context simultaneously.

Think of it as the best of both worlds:

| Model | Strategy | Limitation |
|---|---|---|
| BERT | Masks random tokens; reads bidirectionally | Doesn't model permutation order |
| XLNet | Permutes token order; autoregressive | Slow inference |
| **MPNet** | **Combines both — bidirectional context + permutation awareness** | — |

Training MPNet fuses BERT's **MLM** (Masked Language Modelling) and XLNet's **PLM** (Permuted Language Modelling): tokens are permuted, masked intelligently, and predicted using context that considers both position and dependency. This makes MPNet an excellent backbone for contrastive sentence-similarity training.

---

### all-mpnet-base-v2 Specifically

`all-mpnet-base-v2` is the variant we use. It was created by fine-tuning the pretrained `microsoft/mpnet-base` model on **approximately 1 billion sentence pairs** using a **contrastive learning** objective.

**Contrastive learning in plain English:**
```
Training step:
  Pair A: ("Dog training is fun", "Teaching puppies tricks")
  → These appear together → model learns: make their vectors CLOSE  [similarity ≈ 0.95] ✓

  Random sentence: ("The sky is blue")
  → Doesn't pair with dog sentence → make vectors FAR APART  [similarity ≈ 0.1]  ✓

Repeat for ~1 billion pairs → model gets very good at encoding meaning
```

---

### The 768-Dimensional Output

`all-mpnet-base-v2` produces **768-dimensional embeddings** — one 768-number array per sentence.

Why 768? It's an empirical sweet spot:
- Enough dimensions to capture complex, subtle meaning
- Not so many that computation becomes prohibitively slow
- Powers-of-2-adjacent sizes (512, 768, 1024) tend to work well in Transformer architectures

Imagine each dimension as a dial measuring one aspect of meaning:
```
Dimension   1: "Is this about animals?"        [-1 … +1]
Dimension   2: "Is this about learning?"       [-1 … +1]
Dimension   3: "Positive or negative tone?"    [-1 … +1]
  … 765 more dimensions …
Dimension 768: "Level of formality"            [-1 … +1]
```

Every sentence gets exactly 768 numbers representing its complete meaning.

---

### Sequence Length & Pooling

- Training was limited to **128 tokens** per sequence. Longer inputs are truncated.
- Sentence Transformers use **only the encoder** part of the Transformer, with a **pooling layer** on top to collapse word-level vectors into one sentence vector.

```
Input: "I love my dog"
         ↓
MPNet encodes each token: [I]  [love]  [my]  [dog]
Each token → 768-dim context vector
         ↓
Mean Pooling: average all 4 token vectors dimension-by-dimension
         ↓
Output: ONE 768-dim vector representing the whole sentence
```

---

### Quick Check

> Why would it be bad if MPNet gave the **same embedding** to "bank" in **"river bank"** and **"bank account"**?  
> How would that break semantic search?


In [ ]:

# Section 2 — Load MPNet and inspect embeddings

import numpy as np
from sentence_transformers import SentenceTransformer

print("--- Loading all-mpnet-base-v2 ---")
print("(First run downloads ~420 MB from Hugging Face — please wait)")
print()

try:
    mpnet_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
    print("Model loaded successfully")
except Exception as loading_error:
    print(f"Model loading failed: {loading_error}")
    print("   Check your internet connection and try again.")
    raise

print()

sentences_to_embed = [
    "I love my dog",
    "I adore my puppy",
    "The sky is blue",
]

print("--- Encoding sentences ---")
embeddings_array = mpnet_model.encode(sentences_to_embed)

print()
print(f"Number of sentences encoded : {len(sentences_to_embed)}")
print(f"Shape of embeddings array   : {embeddings_array.shape}")
print(f"Dimensions per sentence     : {embeddings_array.shape[1]}")
print()

for sentence_index, sentence_text in enumerate(sentences_to_embed):
    first_ten_dims = embeddings_array[sentence_index][:10]
    formatted_dims = [f"{value:.3f}" for value in first_ten_dims]
    print(f"Sentence {sentence_index}: \"{sentence_text}\"")
    print(f"  First 10 dims: [{', '.join(formatted_dims)}, ...]")
    print()

print("Key observation:")
print("  Sentences 0 and 1 ('dog' / 'puppy') should have VERY similar first-10 values.")
print("  Sentence 2 ('sky') should look noticeably different.")
print("  (We'll measure this precisely in Section 3.)")


In [ ]:

# Section 2 — Context matters: same word, different meaning

print("--- Context-sensitive embeddings ---")
print()

context_sentences = [
    "I parked my car in the parking lot",
    "I went to the park for a walk",
    "Dogs are allowed in the park",
]

context_embeddings = mpnet_model.encode(context_sentences)

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(context_embeddings)

print("Sentences:")
for sentence_index, sentence_text in enumerate(context_sentences):
    print(f"  [{sentence_index}] {sentence_text}")

print()
print("Cosine similarity matrix (1.0 = identical, 0.0 = unrelated):")
print()

print(f"{'':>4}  {'S0':>6}  {'S1':>6}  {'S2':>6}")
for row_index in range(3):
    row_label = f"S{row_index}"
    row_values = similarity_matrix[row_index]
    formatted_values = [f"{value:.3f}" for value in row_values]
    print(f"{row_label:>4}  {'  '.join(formatted_values)}")

print()
print("Key observation:")
print("  S1 vs S2 (both 'park' as green space) should be MORE similar than S0 vs S1.")
print("  MPNet understands that 'park' means different things in different contexts!")


---
## Section 3 — Measuring Similarity: Cosine Similarity Explained

### The Core Question

We now have two 768-dimensional vectors.  
How do we know whether they represent **similar meanings**?

---

### Cosine Similarity: The Intuition

Cosine similarity measures the **angle** between two vectors — not how long they are, but which **direction** they point.

Imagine 2-D arrows on a piece of paper:
```
Vector A: [1, 2]    →  Arrow pointing northeast
Vector B: [1, 2.1]  →  Arrow pointing almost the same direction
         Angle between A and B: ~2°   →  cos(2°) ≈ 0.999  (very similar!)

Vector C: [0, 1]    →  Arrow pointing straight north
         Angle A to C: ~63°            →  cos(63°) ≈ 0.45  (somewhat similar)

Vector D: [-1, -2]  →  Arrow pointing exactly opposite to A
         Angle A to D: 180°            →  cos(180°) = -1.0  (complete opposite)
```

---

### Why Cosine — Not Straight-Line (Euclidean) Distance?

```
Problem with straight-line distance:
  Vector [1, 1] and [10, 10] point in EXACTLY the same direction (identical meaning)
  But Euclidean distance = √((10-1)² + (10-1)²) ≈ 12.7  → looks far apart!

Solution with cosine:
  Both point at 45° northeast → cos(angle between them) = 1.0 → correctly "identical"

Why: Text MEANING depends on DIRECTION, not vector length.
     Longer documents aren't "more meaningful" — they're just longer.
```

---

### Score Interpretation Guide

```
Score   Meaning for text embeddings
─────   ────────────────────────────────────────────────────────
 1.0    Identical meaning / exact paraphrase
 0.8    Very similar — clearly related
 0.6    Related but distinct aspects of a topic
 0.4    Somewhat related — may share a broad topic
 0.2    Barely related
 0.0    Unrelated (vectors are perpendicular)
-0.5    Somewhat opposing ideas
-1.0    Perfectly opposite (rare in practice)

Typical thresholds for a search engine:
  > 0.70  →  Return as a relevant result
  0.50–0.70 →  Include if no better results exist
  < 0.50  →  Do not return
```

---

### Quick Check

> If two texts have cosine similarity of **0.92**, are they about the same topic?  
> Would you return both in search results?


In [ ]:

# Section 3 — Manual cosine similarity from scratch (no sklearn yet)

import numpy as np

print("--- Manual Cosine Similarity Calculation ---")
print()

vector_a = np.array([1.0, 2.0, 3.0])
vector_b = np.array([1.1, 2.1, 3.0])
vector_c = np.array([5.0, 0.1, 0.2])

dot_ab = float(np.dot(vector_a, vector_b))
dot_ac = float(np.dot(vector_a, vector_c))
print(f"Dot product (A·B): {dot_ab:.4f}  ← vectors A and B")
print(f"Dot product (A·C): {dot_ac:.4f}  ← vectors A and C")
print()

magnitude_a = float(np.linalg.norm(vector_a))
magnitude_b = float(np.linalg.norm(vector_b))
magnitude_c = float(np.linalg.norm(vector_c))
print(f"Magnitude of A: {magnitude_a:.4f}")
print(f"Magnitude of B: {magnitude_b:.4f}")
print(f"Magnitude of C: {magnitude_c:.4f}")
print()

cosine_ab = dot_ab / (magnitude_a * magnitude_b)
cosine_ac = dot_ac / (magnitude_a * magnitude_c)
print(f"Cosine similarity (A vs B): {cosine_ab:.4f}  ← should be close to 1.0")
print(f"Cosine similarity (A vs C): {cosine_ac:.4f}  ← should be much lower")
print()
print("Interpretation:")
if cosine_ab > 0.9:
    print("  A and B → very similar (both about dogs/pets)")
if cosine_ac < 0.5:
    print("  A and C → not similar (different topics)")


In [ ]:

# Section 3 — Real MPNet embeddings + cosine similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("--- Cosine Similarity with Real MPNet Embeddings ---")
print()

comparison_sentences = [
    "I enjoy training my dog",
    "Teaching puppies tricks is fun",
    "The weather is sunny today",
    "Dog obedience classes are helpful",
]

comparison_embeddings = mpnet_model.encode(comparison_sentences)
similarity_matrix = cosine_similarity(comparison_embeddings)

print("Sentences:")
for sentence_index, sentence_text in enumerate(comparison_sentences):
    print(f"  [{sentence_index}] {sentence_text}")

print()
print("Pairwise Cosine Similarity Matrix:")
print()

print(f"{'':>4}  {'[0]':>6}  {'[1]':>6}  {'[2]':>6}  {'[3]':>6}")
for row_index in range(4):
    row_values = similarity_matrix[row_index]
    formatted = [f"{value:.3f}" for value in row_values]
    print(f"[{row_index}]   {'  '.join(formatted)}")

print()
print("Key observations:")
print("  [0] vs [1]: Should be HIGH (dog training / teaching puppies)")
print("  [0] vs [2]: Should be LOW  (dog training / sunny weather)")
print("  [0] vs [3]: Should be MODERATE-HIGH (dog training / obedience classes)")
print()

sim_0_1 = similarity_matrix[0][1]
sim_0_2 = similarity_matrix[0][2]
print(f"  Dog training vs Teaching puppies : {sim_0_1:.3f}")
print(f"  Dog training vs Sunny weather    : {sim_0_2:.3f}")


---
## Section 4 — FAISS: Searching Millions of Vectors Instantly

### The Problem FAISS Solves

```
Without FAISS — naive search over 1 million documents:
  → Compare query vector with ALL 1,000,000 embeddings
  → Calculate cosine similarity for each one
  → Sort and return top 10
  → Time: 10+ seconds  ❌  (way too slow)

With FAISS — smart indexed search:
  → Use clustering to skip irrelevant embeddings
  → Only compare with ~50 candidate vectors
  → Return top 10 in ~10 milliseconds  ✅
  → Speed improvement: ~1,000×
```

---

### What Is FAISS?

**FAISS** (Facebook AI Similarity Search) is an **open-source library for similarity search and clustering of dense vectors**, developed by **Meta AI** (Meta Platforms).

Key facts:
- Initial release: **February 22, 2017** | Stable release: **v1.12.0 (August 12, 2025)**
- Handles sets of vectors of **any size**, including ones that do not fit in RAM
- Enables nearest-neighbor search **8.5× faster** than the previous state-of-the-art
- Meta internally uses FAISS to index **1.5 trillion 144-dimensional vectors**

> FAISS is a **toolkit** — it provides indexing, clustering, compression, and transformation methods for vectors. You pick the right index for your scale and accuracy needs.

---

### How FAISS Works — The Intuition

**Without indexing (naive):**
> Like finding a person in a city by asking every single resident one-by-one.

**With IVF indexing (FAISS):**
> Like using city districts:
> 1. Determine which district the person lives in (based on location)  
> 2. Ask only the residents of that district  
> 3. Found quickly!

FAISS uses a popular approach called the **Inverted File Index (IVF)**: it partitions the embedding space into Voronoi cells (clusters) during index construction, then at query time restricts the search to only the nearest cluster(s).

---

### Three Index Types You Need to Know

| Index | Method | Speed | Accuracy | Memory | When to Use |
|---|---|---|---|---|---|
| `IndexFlatL2` | Compare every vector (exhaustive) | 🐢 Slow | ✅ Exact | Normal | < 100K vectors, accuracy critical |
| `IndexIVFFlat` | Cluster first, search nearby clusters | 🚀 Fast | ~95%+ | Normal | 100K – 10M vectors |
| `IndexIVFPQ` | Cluster + compress with Product Quantization | ⚡ Fastest | Good | 10-100× less | > 10M vectors, memory limited |

**Name decoder:**
- `IVF` = Inverted File (the clustering part)  
- `Flat` = Vectors stored as-is (no compression)  
- `PQ` = Product Quantization (compression)  

---

### FAISS Search Process

```
Step 1: Build index  (one-time, done offline)
  ├─ Take all document embeddings (768-dim each)
  ├─ Cluster them into groups  (IVF)
  ├─ Store organised by cluster
  └─ Index is ready

Step 2: Query  (repeated, at search time)
  ├─ User types: "I love dogs"
  ├─ MPNet: convert to 768-dim query embedding
  ├─ FAISS: which cluster does this query belong to?
  ├─ FAISS: fetch vectors from that cluster only
  ├─ Rank by similarity
  └─ Return top-k documents
```

---

### Memory Estimate

`768 dimensions × num_documents × 4 bytes (float32)`

| Documents | RAM needed |
|---|---|
| 10,000 | ~30 MB |
| 1,000,000 | ~3 GB |
| 1,000,000,000 | ~3 TB → use `IndexIVFPQ` |

---

### Quick Check

> Why is `IndexFlatL2` **perfect** for 1,000 documents but **terrible** for 1 billion documents?


In [ ]:

# Section 4 — IndexFlatL2: the simplest FAISS index

import faiss
import numpy as np

print("--- FAISS IndexFlatL2 (Exhaustive Search) ---")
print()

small_documents = [
    "How to train your dog at home",
    "Best dog breeds for families",
    "Cat care guide for beginners",
    "Tips for teaching puppies tricks",
    "Dog nutrition and feeding schedule",
    "Cooking pasta for dinner",
    "Pet grooming and hygiene tips",
    "Traveling with your dog",
]

print(f"Encoding {len(small_documents)} documents with MPNet...")
small_embeddings = mpnet_model.encode(small_documents)
print(f"Embeddings shape: {small_embeddings.shape}")
print()

embedding_dimension = 768
flat_index = faiss.IndexFlatL2(embedding_dimension)
flat_index.add(small_embeddings.astype('float32'))
print(f"Index type     : {type(flat_index).__name__}")
print(f"Vectors stored : {flat_index.ntotal}")
print()

query_text = "How do I teach my puppy tricks?"
print(f"Query: \"{query_text}\"")
print()

query_embedding = mpnet_model.encode(query_text)
query_2d = query_embedding.astype('float32').reshape(1, -1)

k_results = 3
distances, result_indices = flat_index.search(query_2d, k_results)

print(f"Top {k_results} results (IndexFlatL2 — L2 distance, lower = more similar):")
for rank in range(k_results):
    document_index = result_indices[0][rank]
    l2_distance = distances[0][rank]
    document_text = small_documents[document_index]
    intuitive_score = 1.0 / (1.0 + l2_distance)
    print(f"  Rank {rank + 1} | Score {intuitive_score:.3f} | L2 {l2_distance:.3f} | {document_text}")

print()
print("Note: IndexFlatL2 compares the query with ALL 8 documents — exact but slow at scale.")


In [ ]:

# Section 4 — IndexIVFFlat: clustered search (faster, slightly approximate)

import faiss
import numpy as np
import time

print("--- FAISS IndexIVFFlat (Clustered Search) ---")
print()

num_synthetic_docs = 2000
embedding_dimension = 768

np.random.seed(42)
synthetic_embeddings = np.random.rand(num_synthetic_docs, embedding_dimension).astype('float32')

print(f"Synthetic dataset: {num_synthetic_docs} vectors, each {embedding_dimension} dimensions")
print()

# Rule of thumb: nlist = sqrt(N); FAISS requires at least nlist*39 training points.
# With 2000 docs: sqrt(2000) ≈ 45; 45*39 = 1755 < 2000 — safe.
num_voronoi_cells = 45

quantizer = faiss.IndexFlatL2(embedding_dimension)
ivf_index = faiss.IndexIVFFlat(quantizer, embedding_dimension, num_voronoi_cells)

print("Training IVFFlat index (k-means clustering)...")
ivf_index.train(synthetic_embeddings)
print(f"Is trained: {ivf_index.is_trained}")

ivf_index.add(synthetic_embeddings)
print(f"Vectors in index: {ivf_index.ntotal}")
print()

ivf_index.nprobe = 10
print(f"nprobe (clusters to search): {ivf_index.nprobe}")
print()

flat_benchmark_index = faiss.IndexFlatL2(embedding_dimension)
flat_benchmark_index.add(synthetic_embeddings)

query_vector = np.random.rand(1, embedding_dimension).astype('float32')

start_time_flat = time.time()
flat_distances, flat_indices = flat_benchmark_index.search(query_vector, 10)
elapsed_flat = (time.time() - start_time_flat) * 1000

start_time_ivf = time.time()
ivf_distances, ivf_indices = ivf_index.search(query_vector, 10)
elapsed_ivf = (time.time() - start_time_ivf) * 1000

print("Speed comparison on synthetic 2000-doc dataset:")
print(f"  IndexFlatL2  (exhaustive):  {elapsed_flat:.3f} ms")
print(f"  IndexIVFFlat (clustered) :  {elapsed_ivf:.3f} ms")
print()

flat_result_set = set(flat_indices[0].tolist())
ivf_result_set  = set(ivf_indices[0].tolist())
overlap_count = len(flat_result_set & ivf_result_set)
print(f"Result overlap: {overlap_count}/10 documents match between both indexes")
print("(High overlap = IVFFlat found almost the same results as the exact search)")
print()
print("Key takeaway: IVFFlat trades a tiny bit of accuracy for significant speed gains.")


---
## Section 5 — Building Real Semantic Search with MPNet + FAISS

### The Complete Pipeline

```
1. Collect documents
   └── You have: N articles / FAQ entries / product descriptions

2. Embed (MPNet)
   └── all-mpnet-base-v2 → one 768-dim vector per document

3. Index (FAISS)
   └── Load vectors into IndexFlatL2 (small) or IndexIVFFlat (large)
   └── FAISS organises them for fast nearest-neighbour lookup

4. Search (at query time — repeated)
   └── User query → MPNet → 768-dim query vector
   └── FAISS → top-k most similar document vectors
   └── Return ranked document list with similarity scores
```

---

### Real-World Applications

| Domain | Query Example | Semantic Search Finds |
|---|---|---|
| **E-commerce** | "comfortable shoes for running" | Running shoes, sneakers, athletic footwear |
| **Customer Support** | "How do I reset my password?" | Password reset guide, account recovery FAQ |
| **Research** | "Transfer learning in NLP" | Papers on fine-tuning, domain adaptation, BERT |
| **HR / Recruiting** | "Experience with cloud infrastructure" | Candidates mentioning AWS, Azure, GCP, Kubernetes |

---

### Disclaimers

- `all-mpnet-base-v2` was trained on ~1 billion sentence pairs. Results reflect the training distribution — rare domains may produce weaker embeddings.  
- For production with millions of vectors, prefer `IndexIVFFlat` or `IndexIVFPQ` over `IndexFlatL2`.  
- FAISS is a **toolkit** — it enables similarity search and clustering; it does not store metadata (document text). You must maintain your own document list and use FAISS indices to look up the right item.  
- 768-dim vectors × 4 bytes × N documents = RAM needed. Example: 1M docs ≈ 3 GB. Use `IndexIVFPQ` for larger datasets.


In [ ]:

# Section 5 — Exercise 1: Simple end-to-end semantic search

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print("=== Exercise 1: Simple Semantic Search Pipeline ===")
print()

search_documents = [
    "How to train your dog at home",
    "Best dog breeds for families",
    "Cat care guide for beginners",
    "Tips for teaching puppies tricks",
    "Dog nutrition and feeding schedule",
    "Cooking recipes for a dinner party",
    "Pet grooming and hygiene tips",
    "Traveling with your dog on a plane",
]

print(f"Document collection: {len(search_documents)} items")
print()

print("Embedding documents with all-mpnet-base-v2...")
document_embeddings = mpnet_model.encode(
    search_documents,
    convert_to_numpy=True,
    show_progress_bar=False,
)
print(f"Embeddings shape: {document_embeddings.shape}")
print()

vector_dimension = 768
search_index = faiss.IndexFlatL2(vector_dimension)
search_index.add(document_embeddings.astype('float32'))
print(f"FAISS index ready. Vectors stored: {search_index.ntotal}")
print()

example_queries = [
    "How do I teach my puppy tricks?",
    "Best dog food brands",
    "Where to travel with dogs?",
    "How to cook pasta",
]

print("=" * 60)
for query_text in example_queries:
    print()
    print(f"Query: \"{query_text}\"")

    query_vector = mpnet_model.encode(query_text, convert_to_numpy=True)
    query_2d = query_vector.astype('float32').reshape(1, -1)

    top_k = 3
    result_distances, result_indices = search_index.search(query_2d, top_k)

    print(f"Top {top_k} results:")
    for rank in range(top_k):
        doc_index = result_indices[0][rank]
        l2_dist = float(result_distances[0][rank])
        intuitive_score = 1.0 / (1.0 + l2_dist)
        doc_text = search_documents[doc_index]
        print(f"  {rank + 1}. [{intuitive_score:.3f}] {doc_text}")


---
### Exercise 2 — Index Type Comparison

The cell below creates a **2,000-document synthetic dataset** and measures the speed and accuracy of `IndexFlatL2` versus `IndexIVFFlat`.

> **Note on dataset size:** FAISS requires at least `nlist × 39` training vectors. With `nlist=32` that minimum is 1,248 — so we use 2,000 documents to stay safely above it and avoid warnings.


In [ ]:

# Section 5 — Exercise 2: Comparing index types side by side

import faiss
import numpy as np
import time

print("=== Exercise 2: Index Type Comparison ===")
print()

# FIX: num_docs=2000 so FAISS has enough training points for nlist=32.
# FAISS requires >= nlist*39 vectors; 32*39=1248, so 2000 is safely above the threshold.
num_docs = 2000
vec_dim = 768

np.random.seed(0)
synthetic_vecs = np.random.rand(num_docs, vec_dim).astype('float32')

print(f"Dataset: {num_docs} synthetic documents × {vec_dim} dimensions")
print()

flat_index_cmp = faiss.IndexFlatL2(vec_dim)
flat_index_cmp.add(synthetic_vecs)
print(f"IndexFlatL2  built. Vectors: {flat_index_cmp.ntotal}")

nlist_cmp = 32
quantizer_cmp = faiss.IndexFlatL2(vec_dim)
ivf_index_cmp = faiss.IndexIVFFlat(quantizer_cmp, vec_dim, nlist_cmp)

ivf_index_cmp.train(synthetic_vecs)
ivf_index_cmp.add(synthetic_vecs)
ivf_index_cmp.nprobe = 8
print(f"IndexIVFFlat built. Vectors: {ivf_index_cmp.ntotal}  |  nlist={nlist_cmp}  |  nprobe=8")
print()

np.random.seed(99)
test_query = np.random.rand(1, vec_dim).astype('float32')

flat_start = time.time()
flat_dists_cmp, flat_idxs_cmp = flat_index_cmp.search(test_query, 10)
flat_elapsed_ms = (time.time() - flat_start) * 1000

ivf_start = time.time()
ivf_dists_cmp, ivf_idxs_cmp = ivf_index_cmp.search(test_query, 10)
ivf_elapsed_ms = (time.time() - ivf_start) * 1000

flat_result_set_cmp = set(flat_idxs_cmp[0].tolist())
ivf_result_set_cmp  = set(ivf_idxs_cmp[0].tolist())
overlap_cmp = len(flat_result_set_cmp & ivf_result_set_cmp)

print(f"{'Index':>20} | {'Time (ms)':>10} | {'Accuracy':>12} | {'Use case':>25}")
print("-" * 80)

flat_acc_str = "100% (exact)"
ivf_acc_str = f"{overlap_cmp}/10 results match"

print(f"{'IndexFlatL2':>20} | {flat_elapsed_ms:>10.3f} | {flat_acc_str:>12} | {'< 100K vectors':>25}")
print(f"{'IndexIVFFlat':>20} | {ivf_elapsed_ms:>10.3f} | {ivf_acc_str:>12} | {'100K – 10M vectors':>25}")

print()
print("Key takeaway:")
print("  Both indexes find similar results, but IVFFlat is faster because")
print(f"  it only searches {ivf_index_cmp.nprobe} of {nlist_cmp} clusters instead of all {num_docs} vectors.")


---
### Exercise 3 — What Breaks Semantic Search? Common Issues & Fixes

```
1. PROBLEM: Search returns unrelated documents
   CAUSE:   Using raw L2 distance without normalisation → scores hard to interpret
   FIX:     Convert L2 distance to a 0-1 score: score = 1 / (1 + distance)
            OR use faiss.IndexFlatIP with normalised vectors for cosine search

2. PROBLEM: Searches are slow with IndexFlatL2 on a large dataset
   CAUSE:   No clustering — every query compares all N vectors
   FIX:     Use IndexIVFFlat or IndexIVFPQ for datasets > 100K vectors

3. PROBLEM: Different embeddings for the same sentence each time
   CAUSE:   Model is in training mode (dropout active)
   FIX:     SentenceTransformer sets eval mode automatically — don't mix raw PyTorch

4. PROBLEM: Memory error when loading 1M+ documents
   CAUSE:   Full float32 vectors: 1M × 768 × 4 bytes ≈ 3 GB RAM
   FIX:     Use IndexIVFPQ — Product Quantization compresses vectors 10-100×

5. PROBLEM: Every document gets similarity ≈ 0.99 (everything looks the same)
   CAUSE:   Encoding very short or repetitive texts (no signal)
   FIX:     Use richer, longer document text; ensure documents are meaningfully different

6. PROBLEM: IVFFlat gives bad results (low overlap with FlatL2)
   CAUSE:   nprobe too low — searching too few clusters
   FIX:     Increase nprobe (try 10-20% of nlist); or increase nlist during training

7. PROBLEM: FAISS raises a warning about too few training points
   CAUSE:   Number of training vectors < nlist * 39
   FIX:     Reduce nlist (rule of thumb: sqrt(N)) or increase your dataset size
```


In [ ]:

# Section 5 — Exercise 4: Production-ready SemanticSearchEngine class

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print("=== Exercise 4: SemanticSearchEngine Class ===")
print()


class SemanticSearchEngine:
    """
    A minimal semantic search engine backed by MPNet + FAISS.

    Usage (reusing an already-loaded model — preferred):
        engine = SemanticSearchEngine(documents, model=mpnet_model)
        results = engine.search("my query", k=5)

    Usage (standalone — loads its own model):
        engine = SemanticSearchEngine(documents)
        results = engine.search("my query", k=5)
    """

    def __init__(self, documents,
                 model=None,
                 model_name='sentence-transformers/all-mpnet-base-v2'):
        """
        Parameters
        ----------
        documents  : list of str  — the documents to make searchable
        model      : SentenceTransformer instance or None.
                     Pass the already-loaded mpnet_model to avoid a redundant download.
        model_name : str — Hugging Face model ID used only when model=None.
        """
        self.documents = documents

        # FIX: Accept a pre-loaded model to avoid creating a redundant copy.
        # If no model is supplied, load one (useful when running this class standalone).
        if model is not None:
            self.model = model
            print("Using provided model instance (no re-download needed).")
        else:
            print(f"Loading model '{model_name}' ...")
            self.model = SentenceTransformer(model_name)

        print(f"Embedding {len(documents)} documents...")
        document_embeddings = self.model.encode(
            documents,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        document_embeddings_f32 = document_embeddings.astype('float32')

        vector_dimension = document_embeddings_f32.shape[1]
        self.faiss_index = faiss.IndexFlatL2(vector_dimension)
        self.faiss_index.add(document_embeddings_f32)

        print(f"Index ready. {self.faiss_index.ntotal} vectors, {vector_dimension} dimensions each.")
        print()

    def search(self, query_text, k=5):
        """
        Find the top-k most semantically similar documents.

        Returns list of dicts with keys: rank, similarity, document, document_index.
        """
        query_vector = self.model.encode(query_text, convert_to_numpy=True)
        query_2d = query_vector.astype('float32').reshape(1, -1)

        result_distances, result_indices = self.faiss_index.search(query_2d, k)

        results = []
        for rank in range(k):
            doc_index = int(result_indices[0][rank])
            l2_distance = float(result_distances[0][rank])
            similarity_score = 1.0 / (1.0 + l2_distance)

            results.append({
                'rank': rank + 1,
                'similarity': similarity_score,
                'document': self.documents[doc_index],
                'document_index': doc_index,
            })

        return results


# ── Demonstrate ──────────────────────────────────────────────────

knowledge_base = [
    "MPNet uses two-stream self-attention combining positional and masked context",
    "all-mpnet-base-v2 produces 768-dimensional sentence embeddings",
    "FAISS was developed by Meta AI for billion-scale vector search",
    "Cosine similarity measures the angle between two vectors",
    "Sentence Transformers use only the encoder and a pooling layer",
    "IndexFlatL2 performs exhaustive exact search across all vectors",
    "IndexIVFFlat clusters vectors into Voronoi cells for faster search",
    "Contrastive learning trains MPNet to place similar sentences close together",
    "FAISS can index 1.5 trillion vectors as used internally by Meta",
    "Semantic search finds documents by meaning, not keyword overlap",
]

# Pass the already-loaded mpnet_model to avoid a second download
engine = SemanticSearchEngine(knowledge_base, model=mpnet_model)

demo_queries = [
    "How does all-mpnet-base-v2 create embeddings?",
    "What is FAISS and who made it?",
    "How is similarity between sentences measured?",
]

print("=" * 65)
for demo_query in demo_queries:
    print()
    print(f"Query: \"{demo_query}\"")
    search_results = engine.search(demo_query, k=3)
    for result in search_results:
        print(f"  {result['rank']}. [{result['similarity']:.3f}] {result['document']}")


---
## Summary — What You Mastered Today

| Concept | What You Learned |
|---|---|
| **Keyword search limitations** | Exact word matching misses synonyms, context, and intent |
| **MPNet architecture** | Two-stream self-attention; combines BERT MLM + XLNet PLM |
| **all-mpnet-base-v2** | 768-dim embeddings; trained on ~1B sentence pairs via contrastive learning; 128-token limit |
| **Pooling** | Mean pooling collapses word-level vectors into one sentence vector |
| **Cosine similarity** | Measures the angle between vectors; 1.0 = identical, 0.0 = unrelated |
| **FAISS** | Meta AI's open-source toolkit for billion-scale vector similarity search |
| **IndexFlatL2** | Exact exhaustive search — best for < 100K vectors |
| **IndexIVFFlat** | Clustered approximate search — best for 100K–10M vectors |
| **IndexIVFPQ** | Clustered + compressed — best for > 10M vectors or memory-constrained environments |

---

### Key Performance Facts

- FAISS enables nearest-neighbour search **8.5× faster** than the previous state-of-the-art  
- Meta internally uses FAISS to index **1.5 trillion 144-dimensional vectors**  
- `all-mpnet-base-v2` trained on **~1 billion sentence pairs** — robust out of the box

---

### What's Next?

- **Persistent indexes:** Save your FAISS index to disk with `faiss.write_index()` and reload with `faiss.read_index()`  
- **Larger models:** Explore `all-roberta-large-v1` (1024 dims) for higher accuracy  
- **GPU FAISS:** Replace `faiss-cpu` with `faiss-gpu` for 10–100× faster indexing  
- **Hybrid search:** Combine semantic search with BM25 keyword scores (reciprocal rank fusion)  
- **Metadata filtering:** Pair FAISS with a metadata store (e.g. SQLite) to filter by date, category, etc.

---

> **Congratulations!** You can now build production-aware semantic search systems without relying on any external API. Every component — MPNet embeddings and FAISS indexing — runs entirely on your own hardware.
